In [20]:
"""
=========================================================
05_train_model.py
=========================================================

Purpose
-------
Train the CatBoost model using the best hyperparameters
found during Optuna tuning.

Inputs
------
data/processed/feature_engineered.csv
outputs/selected_features.json
outputs/best_hyperparameters.json

Outputs
-------
models/customer_propensity_model.cbm
outputs/model_metrics.json
outputs/feature_importance.csv
"""

'\n=========================================================\n05_train_model.py\n=========================================================\n\nPurpose\n-------\nTrain the CatBoost model using the best hyperparameters\nfound during Optuna tuning.\n\nInputs\n------\ndata/processed/feature_engineered.csv\noutputs/selected_features.json\noutputs/best_hyperparameters.json\n\nOutputs\n-------\nmodels/customer_propensity_model.cbm\noutputs/model_metrics.json\noutputs/feature_importance.csv\n'

In [23]:
import json
import warnings
import os
import joblib
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from sklearn.model_selection import train_test_split
from pathlib import Path
warnings.filterwarnings("ignore")

In [ ]:
# ==========================================================
# Paths
# ==========================================================

BASE_DIR = Path.cwd().parent

os.makedirs("models", exist_ok=True)

INPUT_FILE = BASE_DIR /  "data" / "feature_engineered.csv"
FIGURE_PATH = BASE_DIR /"outputs"
FIGURE_PATH_model = BASE_DIR /"models"

In [25]:
#Global declare random state
RANDOM_STATE = 42

In [26]:
print("=" * 70)
print("CUSTOMER PROPENSITY MODEL TRAINING")
print("=" * 70)

CUSTOMER PROPENSITY MODEL TRAINING


In [27]:
# ==========================================================
# LOAD DATA
# ==========================================================

print("=" * 60)
print("Loading Dataset")
print("=" * 60)

df = pd.read_csv(INPUT_FILE)

with open(f"{FIGURE_PATH}/selected_features.json") as f:
    FEATURES = json.load(f)

with open(f"{FIGURE_PATH}/best_hyperparameters.json") as f:
    BEST_PARAMS = json.load(f)

TARGET = "target"

X = df[FEATURES].copy()
y = df[TARGET]

Loading Dataset


In [28]:
# ==========================================================
# CATEGORICAL FEATURES
# ==========================================================

categorical_columns = [
    "Category_Bucket_final",
    "Vertical",
    "self_dealer_status",
    "Dealing_Zone",
    "plan"
]

for col in categorical_columns:
    if col in X.columns:
        X[col] = X[col].fillna("Missing").astype(str)

cat_features = [
    X.columns.get_loc(col)
    for col in categorical_columns
    if col in X.columns
]

In [29]:
# ==========================================================
# TRAIN / VALIDATION / TEST SPLIT
# ==========================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print(f"\nTraining Samples   : {len(X_train):,}")
print(f"Validation Samples : {len(X_valid):,}")
print(f"Test Samples       : {len(X_test):,}")


Training Samples   : 350,000
Validation Samples : 75,000
Test Samples       : 75,000


In [30]:
# ==========================================================
# MODEL
# ==========================================================

BEST_PARAMS.update({

    "loss_function": "Logloss",

    "eval_metric": "AUC",

    "random_seed": RANDOM_STATE,

    "verbose": 100

})

model = CatBoostClassifier(**BEST_PARAMS)

In [31]:
# ==========================================================
# TRAIN
# ==========================================================

print("\nTraining CatBoost...\n")

model.fit(

    X_train,
    y_train,

    cat_features=cat_features,

    eval_set=(X_valid, y_valid),

    use_best_model=True

)


Training CatBoost...

0:	test: 0.5802213	best: 0.5802213 (0)	total: 1.2s	remaining: 10m 57s
100:	test: 0.5914863	best: 0.5914863 (98)	total: 23.7s	remaining: 1m 45s
200:	test: 0.5977871	best: 0.5980257 (168)	total: 53.5s	remaining: 1m 32s
300:	test: 0.5986728	best: 0.6012457 (236)	total: 1m 28s	remaining: 1m 12s
400:	test: 0.5941369	best: 0.6012457 (236)	total: 2m 4s	remaining: 46.1s
500:	test: 0.5920367	best: 0.6012457 (236)	total: 2m 41s	remaining: 15.4s
548:	test: 0.5885839	best: 0.6012457 (236)	total: 3m	remaining: 0us

bestTest = 0.6012457487
bestIteration = 236

Shrink model to first 237 iterations.


CatBoostClassifier(bagging_temperature=0.9953284948454381, border_count=172, depth=6, eval_metric='AUC', iterations=549, l2_leaf_reg=2, learning_rate=0.09009468297411623, loss_function='Logloss', random_seed=42, random_strength=4.412078268113714, verbose=100)

In [32]:
# ==========================================================
# EVALUATION
# ==========================================================

def evaluate(name, X_data, y_data):

    prob = model.predict_proba(X_data)[:, 1]

    pred = (prob >= 0.5).astype(int)

    roc = roc_auc_score(y_data, prob)

    pr = average_precision_score(y_data, prob)

    gini = (2 * roc) - 1

    print("\n" + "=" * 60)

    print(name.upper())

    print("=" * 60)

    print(f"ROC AUC : {roc:.6f}")

    print(f"PR AUC  : {pr:.6f}")

    print(f"Gini    : {gini:.6f}")

    print("\nConfusion Matrix")

    print(confusion_matrix(y_data, pred))

    print("\nClassification Report")

    print(classification_report(y_data, pred))

    return roc, pr, gini

train_metrics = evaluate("Train", X_train, y_train)

valid_metrics = evaluate("Validation", X_valid, y_valid)

test_metrics = evaluate("Test", X_test, y_test)


TRAIN
ROC AUC : 0.659469
PR AUC  : 0.033777
Gini    : 0.318938

Confusion Matrix
[[347294      0]
 [  2706      0]]

Classification Report
              precision    recall  f1-score   support

           0       0.99      1.00      1.00    347294
           1       0.00      0.00      0.00      2706

    accuracy                           0.99    350000
   macro avg       0.50      0.50      0.50    350000
weighted avg       0.98      0.99      0.99    350000


VALIDATION
ROC AUC : 0.601246
PR AUC  : 0.010465
Gini    : 0.202491

Confusion Matrix
[[74420     0]
 [  580     0]]

Classification Report
              precision    recall  f1-score   support

           0       0.99      1.00      1.00     74420
           1       0.00      0.00      0.00       580

    accuracy                           0.99     75000
   macro avg       0.50      0.50      0.50     75000
weighted avg       0.98      0.99      0.99     75000


TEST
ROC AUC : 0.598773
PR AUC  : 0.010919
Gini    : 0.197547

C

In [33]:
# ==========================================================
# FEATURE IMPORTANCE
# ==========================================================

importance = pd.DataFrame({

    "Feature": FEATURES,

    "Importance": model.get_feature_importance()

}).sort_values(

    by="Importance",

    ascending=False

)

print("\n")

print("=" * 60)

print("FEATURE IMPORTANCE")

print("=" * 60)

print(importance)

importance.to_csv(

    f"{FIGURE_PATH}/feature_importance.csv",

    index=False

)



FEATURE IMPORTANCE
                          Feature  Importance
13                   never_traded   17.751640
1                  vintage_months   16.989713
4                    ord_cash_log   14.670515
0                     current_age    9.029504
3                Total_Margin_log    7.407357
12                        recency    7.296051
9   Number_of_competition_account    5.741417
5                    ord_derv_log    4.497404
11                   Dealing_Zone    4.342928
2                    DP_Value_log    4.004023
8              self_dealer_status    3.263885
7                        Vertical    2.006176
10                           plan    1.662957
6           Category_Bucket_final    1.336432


In [34]:
# ==========================================================
# SAVE METRICS
# ==========================================================

metrics = {

    "Train ROC AUC": train_metrics[0],

    "Validation ROC AUC": valid_metrics[0],

    "Test ROC AUC": test_metrics[0],

    "Train PR AUC": train_metrics[1],

    "Validation PR AUC": valid_metrics[1],

    "Test PR AUC": test_metrics[1],

    "Train Gini": train_metrics[2],

    "Validation Gini": valid_metrics[2],

    "Test Gini": test_metrics[2]

}

with open(

    f"{FIGURE_PATH}/model_metrics.json",

    "w"

) as f:

    json.dump(

        metrics,

        f,

        indent=4

    )

In [35]:
# ==========================================================
# SAVE MODEL
# ==========================================================

model.save_model(

    f"{FIGURE_PATH_model}/customer_propensity_model.cbm"

)

joblib.dump(

    FEATURES,

    f"{FIGURE_PATH_model}/model_features.pkl"

)

print("\n")

print("=" * 70)

print("MODEL SAVED SUCCESSFULLY")

print("=" * 70)

print(f"Model      : {FIGURE_PATH_model}/customer_propensity_model.cbm")

print(f"Features   : {FIGURE_PATH_model}/model_features.pkl")

print(f"Metrics    : {FIGURE_PATH}/model_metrics.json")

print(f"Importance : {FIGURE_PATH}/feature_importance.csv")



MODEL SAVED SUCCESSFULLY
Model      : models/customer_propensity_model.cbm
Features   : models/model_features.pkl
Metrics    : outputs/model_metrics.json
Importance : outputs/feature_importance.csv
